# 📖 Notebook 4: Advanced Gateway Patterns

You've seen the core gateway responsibilities (routing, rate limiting, auth, transformation). This notebook covers the **advanced patterns** that turn a basic gateway into a production-grade one:

| Pattern | What it solves |
|---------|----------------|
| **Circuit-breaker-style failover** | Stops sending traffic to a backend that keeps failing |
| **Request aggregation (BFF)** | Lets clients make one call instead of many |
| **CORS** | Lets browsers from other origins call your API safely |
| **Observability (logging/tracing)** | Makes it possible to debug what went wrong, in which service |
| **SSL/TLS termination** | Terminates HTTPS at the edge so backends stay simple |

We'll follow the same 🚫 BAD → ✅ BETTER → 🏆 BEST structure wherever it fits.

## Learning Objectives

By the end of this notebook, you'll understand:
- Why you need failover and what nginx can (and cannot) do for you
- The difference between true circuit breakers and passive failover
- When to aggregate requests at the gateway vs. in a dedicated BFF
- How CORS preflight requests work and how the gateway handles them
- How `X-Request-ID` enables distributed tracing across services
- Why almost every production gateway terminates TLS


## 🛠️ Setup

Make sure infrastructure is running:

```bash
cd 05-microservices/api-gateway
docker-compose up -d --build
```

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".


In [ ]:
import requests
import json
import time

GATEWAY = "http://localhost:8080"

def show(response):
    print(f"Status: {response.status_code}")
    try:
        print(json.dumps(response.json(), indent=2))
    except Exception:
        print(response.text[:300])

try:
    r = requests.get(f"{GATEWAY}/health", timeout=3)
    print(f"✅ API Gateway: {r.json()['status']}")
except Exception as e:
    print(f"❌ API Gateway not running: {e}")
    print("   Run: cd 05-microservices/api-gateway && docker-compose up -d --build")


---

## 1️⃣ Circuit-Breaker-Style Failover

### The problem

One of your backend instances starts failing (network glitch, OOM, bad deploy). If the gateway keeps sending it traffic, every request to that instance fails. Worse: the gateway itself slows down while it waits for timeouts. This is called a **cascading failure**.

### 🚫 BAD: No failover

With a naïve load balancer, 50% of your traffic goes to the dead instance → 50% of users see errors.

### ✅ BETTER: Passive failover (what nginx gives us)

Our `nginx.conf` declares each upstream with:

```nginx
upstream user_backend {
    server user-service-1:5000 max_fails=3 fail_timeout=30s;
    server user-service-2:5000 max_fails=3 fail_timeout=30s;
}

proxy_next_upstream       error timeout http_502 http_503 http_504;
proxy_next_upstream_tries 2;
proxy_connect_timeout     2s;
proxy_read_timeout        5s;
```

Behavior:
- If a backend fails/times out, nginx retries the **next** healthy instance automatically
- After 3 failures in 30s, nginx marks the instance "down" and stops sending it traffic
- After 30s nginx tries again (very coarse "recovery" check)

### 🏆 BEST: True circuit breaker (what you'd use in production)

A real circuit breaker has three states:

```
     closed  ───failures exceed threshold──▶   open
       ▲                                         │
       │                                         │ wait timeout
       │                                         ▼
   success ◀────probe succeeds────   half-open
```

- **closed** — normal traffic flow
- **open** — all requests fail fast (no backend call made at all)
- **half-open** — let a few probe requests through; if they succeed, close; if they fail, re-open

Production tools that implement this: **Envoy**, **Istio**, **Linkerd**, **Resilience4j** (Java), **Polly** (.NET), **pybreaker** (Python). nginx OSS does not.

Let's simulate the pattern in Python so you understand the state machine:


In [ ]:
# A minimal circuit breaker — the pattern your service mesh implements for you.
import time

class CircuitBreaker:
    CLOSED, OPEN, HALF_OPEN = "closed", "open", "half_open"

    def __init__(self, fail_threshold=3, recovery_seconds=5):
        self.state = self.CLOSED
        self.failures = 0
        self.opened_at = 0.0
        self.fail_threshold = fail_threshold
        self.recovery_seconds = recovery_seconds

    def call(self, fn, *args, **kwargs):
        # If OPEN, short-circuit until the recovery window passes
        if self.state == self.OPEN:
            if time.time() - self.opened_at >= self.recovery_seconds:
                self.state = self.HALF_OPEN  # let a probe request through
                print("   ↩️  HALF_OPEN: sending a probe request...")
            else:
                raise RuntimeError("circuit OPEN — failing fast")

        try:
            result = fn(*args, **kwargs)
        except Exception as exc:
            self.failures += 1
            if self.state == self.HALF_OPEN or self.failures >= self.fail_threshold:
                self.state = self.OPEN
                self.opened_at = time.time()
                print(f"   🔥 circuit OPENED after failure: {exc}")
            raise
        else:
            if self.state == self.HALF_OPEN:
                print("   ✅ probe succeeded — circuit CLOSED")
            self.state = self.CLOSED
            self.failures = 0
            return result

# Simulate: a backend that fails 5 times, then recovers
attempts = {"n": 0}
def flaky_backend():
    attempts["n"] += 1
    if attempts["n"] <= 5:
        raise RuntimeError("backend down")
    return "ok"

cb = CircuitBreaker(fail_threshold=3, recovery_seconds=2)
for i in range(12):
    try:
        out = cb.call(flaky_backend)
        print(f"Request {i+1}: state={cb.state:<10} result={out}")
    except Exception as exc:
        print(f"Request {i+1}: state={cb.state:<10} error={exc}")
    time.sleep(0.6)


### What nginx gives us vs. what a service mesh gives us

| Capability | nginx OSS (`max_fails`/`proxy_next_upstream`) | Envoy / Istio |
|-----------|:---:|:---:|
| Retry next instance on error | ✅ | ✅ |
| Eject failing instance temporarily | ✅ (coarse) | ✅ (fine-grained) |
| Open / Half-Open / Closed state machine | ❌ | ✅ |
| Fail-fast when circuit is open | ❌ | ✅ |
| Health checks (active) | ⚠️ nginx Plus only | ✅ |
| Metrics for breaker state | ❌ | ✅ |

**Takeaway:** for a small system, `max_fails` + timeouts is often enough. When you grow to many services and need production-grade resilience, move to a service mesh or a dedicated client-side breaker library.


---

## 2️⃣ Request Aggregation (Backend-For-Frontend pattern)

### The problem

A mobile "profile screen" needs: the user's details **and** their orders. Those live in two services.

### 🚫 BAD: The client makes N calls

```
Mobile app                Network
────────                  ───────
GET /api/users/1    ──────▶   (round-trip #1, ~200ms on mobile)
GET /api/orders?user_id=1 ─▶  (round-trip #2, ~200ms on mobile)
```

Two round-trips means twice the latency — painful on mobile networks — and forces the client to handle partial failures.

### 🏆 BEST: The gateway (or BFF) aggregates

```
Mobile app        Gateway / BFF          Services
────────          ─────────────          ────────
GET /api/profile/1 ─▶ fetch user ──────▶ user-service
                      fetch orders ────▶ order-service
                   ◀── combined JSON
```

One round-trip, one combined payload, and partial failures are handled server-side.

Let's hit the aggregated endpoint:


In [ ]:
time.sleep(1)  # avoid rate limiting from earlier notebooks

print("🏆 Aggregated profile endpoint")
print("=" * 55)
print("ONE call to the gateway returns user + orders:")
print()

r = requests.get(f"{GATEWAY}/api/profile/1")
show(r)


In [ ]:
# Compare: the client-side approach (what you'd have WITHOUT aggregation)
def client_side_composition(user_id):
    r1 = requests.get(f"{GATEWAY}/api/users/{user_id}")
    r2 = requests.get(f"{GATEWAY}/api/orders", params={"user_id": user_id})
    return {"user": r1.json(), "orders": r2.json().get("orders", [])}

def gateway_composition(user_id):
    return requests.get(f"{GATEWAY}/api/profile/{user_id}").json()

N = 10

t0 = time.time()
for _ in range(N):
    client_side_composition("1")
client_ms = (time.time() - t0) / N * 1000

time.sleep(1)

t0 = time.time()
for _ in range(N):
    gateway_composition("1")
gateway_ms = (time.time() - t0) / N * 1000

print(f"Client-side (2 calls/req):  avg {client_ms:.1f} ms per profile")
print(f"Gateway-side (1 call/req):  avg {gateway_ms:.1f} ms per profile")
print()
print("💡 On localhost the gap is small, but on mobile each saved round-trip")
print("   is typically 100–300ms. The savings compound with more sub-calls.")


### Where should composition live?

| Option | Pros | Cons |
|--------|------|------|
| **In the gateway itself** (nginx + Lua / Kong) | One hop, centralized | Gateway becomes smart/stateful |
| **Dedicated BFF service** (one per client type) | Tailored per client (iOS vs Web) | Another service to own |
| **GraphQL gateway** (Apollo, Hasura) | Clients request exactly the fields they need | Schema/tooling complexity |
| **Inside a backend service** (what this lab does) | Simplest | Violates single-responsibility |

For this lab, the aggregation lives in `user_service.py` to keep the container count low. In a real system, prefer a **dedicated BFF** or a **GraphQL gateway** — especially if different client types (iOS/Android/Web) need different payloads.

### Partial-failure behavior

Notice the aggregated endpoint uses a short (1.5s) timeout when calling order-service. If that call fails, the endpoint still returns user data with `orders_unavailable: true`. This **graceful degradation** is a must-have pattern in any aggregation layer — one slow dependency should never take down the whole response.


---

## 3️⃣ CORS (Cross-Origin Resource Sharing)

### The problem

Your web app is served from `https://app.example.com`. Its JavaScript tries to call `https://api.example.com/users`. By default, browsers **block** this — it's a "cross-origin" request. This is a security feature called the **Same-Origin Policy**.

To allow it, the API must respond with CORS headers:

```
Access-Control-Allow-Origin: https://app.example.com
Access-Control-Allow-Methods: GET, POST
Access-Control-Allow-Headers: Content-Type, X-API-Key
```

### Preflight requests

For any request that is **not** a "simple" GET/HEAD/POST-with-simple-headers, the browser first sends an `OPTIONS` request (called a **preflight**) to ask "may I?". If the preflight response has the right CORS headers, the browser sends the real request.

```
Browser              Gateway               Backend
───────              ───────               ───────
OPTIONS /api/...  ──▶                           (never reaches backend)
                     Access-Control-Allow-*
               ◀──── 204 No Content

GET /api/...      ──▶                     ──▶   real request
               ◀──── 200 + CORS headers   ◀──
```

### 🚫 BAD: Make every backend implement CORS

Duplicated headers in every service, inconsistent rules.

### 🏆 BEST: Handle CORS at the gateway

Our nginx config does this for `/api/cors/users`:

```nginx
location /api/cors/users {
    if ($request_method = OPTIONS) {
        add_header Access-Control-Allow-Origin  "*"                              always;
        add_header Access-Control-Allow-Methods "GET, POST, PUT, DELETE, OPTIONS" always;
        add_header Access-Control-Allow-Headers "Content-Type, X-API-Key"        always;
        return 204;
    }
    add_header Access-Control-Allow-Origin "*" always;
    proxy_pass http://user_backend/users;
}
```

Let's watch both halves of the exchange:


In [ ]:
print("🔎 CORS preflight (OPTIONS request)")
print("=" * 55)
r = requests.options(f"{GATEWAY}/api/cors/users",
    headers={
        "Origin": "https://app.example.com",
        "Access-Control-Request-Method": "GET",
        "Access-Control-Request-Headers": "X-API-Key",
    },
    timeout=3,
)
print(f"Status: {r.status_code}  (204 = OK, no body)")
for k, v in r.headers.items():
    if k.lower().startswith("access-control"):
        print(f"  {k}: {v}")

print()
print("🔎 Real request (GET)")
print("=" * 55)
r = requests.get(f"{GATEWAY}/api/cors/users",
                 headers={"Origin": "https://app.example.com"}, timeout=3)
print(f"Status: {r.status_code}")
for k, v in r.headers.items():
    if k.lower().startswith("access-control"):
        print(f"  {k}: {v}")


### Production CORS tips

- **Don't use `Access-Control-Allow-Origin: *` for credentialed requests.** If the browser sends cookies or auth headers, you must echo the exact origin back (and add `Access-Control-Allow-Credentials: true`).
- **Keep an allow-list of origins** instead of `*`. nginx `map` works well for this.
- **Set `Access-Control-Max-Age`** so browsers cache the preflight answer (reduces OPTIONS traffic).
- **Expose response headers** that JS needs to read (like `X-Request-ID`) via `Access-Control-Expose-Headers`.


---

## 4️⃣ Observability: Logging, Tracing, Monitoring

When a user says *"the app was slow yesterday"*, you need to reconstruct what happened across your services. You need **observability**:

- **Logs** — discrete events ("request X returned 500")
- **Metrics** — numbers over time ("p99 latency = 320ms")
- **Traces** — how a single request flowed through every service

The gateway is the perfect place to emit observability signals: **every** request passes through it.

### Structured access logs

Our `nginx.conf` writes JSON access logs to stdout:

```nginx
log_format gateway_json escape=json
    '{"time":"$time_iso8601",'
    '"request_id":"$request_id",'
    '"method":"$request_method",'
    '"uri":"$request_uri",'
    '"status":$status,'
    '"upstream_addr":"$upstream_addr",'
    '"upstream_response_time":"$upstream_response_time",'
    '"request_time":$request_time}';
access_log /dev/stdout gateway_json;
```

To watch the logs live while you use the notebook, open a second terminal:

```bash
docker logs -f api-gateway
```

### Distributed tracing with `X-Request-ID`

The gateway mints a unique ID for every request (nginx's built-in `$request_id`) and forwards it as `X-Request-ID`. If every service logs that ID, you can **grep across all logs** to reconstruct the journey of one request.


In [ ]:
import uuid

print("📝 Sending requests so the gateway writes access logs...")
correlation_tag = str(uuid.uuid4())[:8]  # our own tag (separate from X-Request-ID)

for i in range(5):
    r = requests.get(f"{GATEWAY}/api/users",
                     headers={"User-Agent": f"traffic-bot/{correlation_tag}"})
    print(f"  req {i+1}: status={r.status_code}")

print()
print("Run this in a terminal to see the structured JSON logs for our requests:")
print()
print(f"  docker logs api-gateway 2>&1 | grep {correlation_tag}")
print()
print("You'll see one JSON line per request with: request_id, status,")
print("upstream_addr (which backend served it), and request_time.")


In [ ]:
print("🧵 End-to-end trace ID propagation")
print("=" * 55)

for i in range(3):
    r = requests.get(f"{GATEWAY}/api/debug/headers")
    received = r.json()["received_headers"]
    gw_id = received.get("X-Request-Id")
    served_by = r.json()["served_by"]
    print(f"  req {i+1}: request_id={gw_id}  served_by={served_by}")

print()
print("💡 In production: log this ID in every service, then search by it in")
print("   Kibana/Datadog/Loki to see the entire trace in a single query.")
print()
print("🔗 Real tracing systems (OpenTelemetry, Jaeger, Zipkin) go further:")
print("   they correlate spans across services and show a visual timeline.")


### Metrics to collect at the gateway

If you add nothing else to your stack, these four gateway metrics give you 90% of the value:

1. **RPS per route** — how many requests per second per endpoint
2. **Error rate per route** — percentage of 4xx/5xx responses
3. **Latency per route** — p50, p95, p99 of `$request_time`
4. **Upstream health** — how often each backend instance failed (`$upstream_status`)

Tools that ingest nginx logs/metrics easily: **Prometheus** (via `nginx-prometheus-exporter`), **Grafana**, **Datadog**, **New Relic**, **CloudWatch**.


---

## 5️⃣ SSL / TLS Termination (concept-only in this lab)

Almost every production API is served over HTTPS. The gateway is where TLS is typically **terminated**: the client talks HTTPS to the gateway, and the gateway talks plain HTTP to internal services on a private network.

```
Client ──── HTTPS (TLS 1.3) ────▶ Gateway ──── HTTP (plain) ────▶ Backend
                                   │
                                   ├── Certificate management (cert renewal, rotation)
                                   ├── TLS version & cipher enforcement
                                   └── HSTS, HTTP → HTTPS redirect
```

### 🚫 BAD: Terminate TLS in every backend

Each service needs certs, cert rotation scripts, TLS library upgrades. Cert management becomes your #1 operational nightmare.

### 🏆 BEST: Terminate once at the gateway

The gateway owns certs. Backends stay simple. Example `nginx.conf` (this is in our file as a **commented reference** — the lab doesn't ship certs so it can't run):

```nginx
server {
    listen 443 ssl http2;
    server_name api.example.com;

    ssl_certificate     /etc/ssl/certs/api.example.com.pem;
    ssl_certificate_key /etc/ssl/private/api.example.com.key;

    ssl_protocols       TLSv1.2 TLSv1.3;
    ssl_ciphers         HIGH:!aNULL:!MD5;
    ssl_prefer_server_ciphers on;

    add_header Strict-Transport-Security "max-age=31536000; includeSubDomains" always;

    location /api/users {
        proxy_pass http://user_backend/users;   # plain HTTP internally
    }
}

# Redirect any accidental plain HTTP to HTTPS
server {
    listen 80;
    server_name api.example.com;
    return 301 https://$host$request_uri;
}
```

### Where to get certs

- **Let's Encrypt** — free, automated, 90-day certs (use `certbot`)
- **AWS ACM / GCP / Azure** — free when used with their load balancers
- **Your internal CA** — for private APIs

### mTLS — when the gateway also verifies the client

Standard TLS verifies the **server**. In high-security setups (service-to-service traffic, B2B APIs), **mutual TLS (mTLS)** also verifies the **client's certificate**. nginx supports this via `ssl_client_certificate` + `ssl_verify_client on`. This is common inside service meshes (Istio enables it automatically between services).


---

## 📚 Summary

### What We Learned

| Pattern | Core idea | nginx primitive | Production upgrade |
|---------|----------|-----------------|---------------------|
| Circuit-breaker-style failover | Don't send traffic to a dead instance | `max_fails`, `fail_timeout`, `proxy_next_upstream` | Envoy / Istio / Resilience4j |
| Request aggregation (BFF) | One client call = many backend calls | URL route → service that composes | Dedicated BFF / GraphQL gateway |
| CORS | Let other-origin browsers call your API | `add_header Access-Control-*` | Same, plus origin allow-list |
| Observability | Logs + traces correlated by request ID | `log_format` + `$request_id` | OpenTelemetry / Jaeger / Datadog |
| TLS termination | Encrypt at the edge, plain internally | `listen 443 ssl; ssl_certificate ...` | Managed certs (ACM / Let's Encrypt) |

### Key Takeaways

1. Basic gateways (nginx OSS) handle *most* production needs: routing, LB, rate limiting, auth, CORS, TLS, basic failover. Reach for a service mesh when you need true circuit breakers, fine-grained retries, or mTLS by default.
2. Aggregation at the gateway is a latency win on mobile. Just handle partial failures gracefully.
3. Observability is worthless without **correlation**: a `request_id` logged in every hop is the single most useful thing you can add.
4. TLS belongs at the edge. Backends shouldn't know about certificates.

### Complete Gateway Capability Matrix (all 4 notebooks)

| Notebook | Capability |
|----------|-----------|
| 1 | Path-based routing, load balancing, health checks |
| 2 | Rate limiting, API key authentication |
| 3 | Header injection, API versioning, URL rewriting |
| 4 | Circuit-breaker-style failover, request aggregation (BFF), CORS, observability, TLS termination |

That's the complete picture of what an API gateway does in production. 🎉
